In [ ]:
import os
import io
import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

import ee
import geemap

In [ ]:
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

In [ ]:
# ---- Project paths (matches your repo layout) -------------------------------
PROJECT_ROOT = Path.home() / "Documents" / "Projects" / "Fire_emission_rivers"
DATA_DIR     = PROJECT_ROOT / "data" / "Godavari_River"
OUT_DIR      = PROJECT_ROOT / "outputs" / "Godavari_River"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Basin inputs -------------------------------------------------------------
BASIN_GPKG = DATA_DIR / "Godavari_Basin.gpkg"      # preferred
BASIN_SHP  = DATA_DIR / "Godavari_Shape.shp"        # fallback


In [ ]:

# Load basin boundary
def load_basin_boundary():
    if BASIN_GPKG.exists():
        gdf = gpd.read_file(BASIN_GPKG)
    elif BASIN_SHP.exists():
        gdf = gpd.read_file(BASIN_SHP)
    else:
        raise FileNotFoundError(f"No basin boundary found at {BASIN_GPKG} or {BASIN_SHP}")

    gdf = gdf.to_crs(epsg=4326)
    gdf["dissolve_key"] = "Godavari"
    basin_gdf = gdf.dissolve(by="dissolve_key").reset_index(drop=True)
    basin_gdf["basin_name"] = "Godavari"
    return basin_gdf

basin_gdf = load_basin_boundary()
basin_bbox = tuple(basin_gdf.total_bounds)  # (minx, miny, maxx, maxy)
print("Basin bbox:", basin_bbox)
basin_gdf.plot(edgecolor="black", facecolor="tan", alpha=0.5, figsize=(6, 6))
plt.title("Godavari Basin Boundary")
plt.show()

### FIRMS Fire Activity Dataset

In [ ]:
# ---- NASA FIRMS API Config ----------------------------------------------------
# Get a free MAP KEY at: https://firms.modaps.eosdis.nasa.gov/api/map_key/
from dotenv import load_dotenv

load_dotenv()  # reads .env from the current working directory (your project root)
FIRMS_MAP_KEY = os.environ["FIRMS_MAP_KEY"]

if not FIRMS_MAP_KEY:
    raise ValueError("FIRMS_MAP_KEY not found - check your .env file")

START_YEAR = 2019
END_YEAR   = 2024

# Archive ("Standard Processing") sources - combine MODIS + VIIRS for full FRP coverage
FIRMS_SOURCES = [
    "MODIS_SP",          # Covers 2019–2024 fully (1km resolution)
    "VIIRS_SNPP_SP",     # Covers 2019–2024 fully (375m resolution)
    "VIIRS_NOAA20_SP"    # Covers 2019–2024 fully (375m resolution)
]

In [ ]:
'''import requests

# 1. Insert your actual key from https://firms.modaps.eosdis.nasa.gov/api/map_key/
TEST_KEY = os.environ["FIRMS_MAP_KEY"]  

# Test 5 days in 2023 for Godavari bbox
test_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{TEST_KEY}/VIIRS_SNPP_SP/77.0,16.0,82.0,20.0/5/2023-04-01"

resp = requests.get(test_url)
print("Status Code:", resp.status_code)
print("Response Header/First 300 chars:")
print(resp.text[:300])'''

In [ ]:
# -------------------------------------------------------------------
# 1. fetch_firms_year(source, year, bbox, map_key)
# -------------------------------------------------------------------
# - Breaks 1 calendar year into 10-day start dates to respect FIRMS API single-query limits.
# - Sends HTTP GET requests for each 10-day chunk and parses valid CSV responses into DataFrames.
# - Combines the chunks into a single year-long DataFrame and tags it with the data source name.

def fetch_firms_year(source, year, bbox, map_key):
    """Fetch one calendar year of FIRMS archive detections by chunking in 10-day steps."""
    min_lon, min_lat, max_lon, max_lat = bbox
    area_str = f"{min_lon},{min_lat},{max_lon},{max_lat}"
    
    # 1. Break the entire year into 10-day starting dates
    start_dates = pd.date_range(start=f"{year}-01-01", end=f"{year}-12-31", freq="5D")
    year_dfs = []

    # 2. Loop through each 10-day chunk
    for s_date in start_dates:
        days_remaining = (pd.Timestamp(f"{year}-12-31") - s_date).days + 1
        day_range = min(5, days_remaining)  # Strictly kept under FIRMS 5-day API limit
        date_str = s_date.strftime("%Y-%m-%d")

        url = (
            f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
            f"{map_key}/{source}/{area_str}/{day_range}/{date_str}"
        )

        try:
            resp = requests.get(url, timeout=60)
            if resp.status_code == 200 and not resp.text.strip().lower().startswith("invalid"):
                chunk_df = pd.read_csv(io.StringIO(resp.text))
                if not chunk_df.empty:
                    year_dfs.append(chunk_df)
        except Exception as e:
            print(f"  [warn] Error fetching {source} for {date_str}: {e}")

        time.sleep(0.1)  # Short pause to protect rate limits

    if year_dfs:
        df = pd.concat(year_dfs, ignore_index=True).drop_duplicates()
        df["source"] = source
        return df

    return pd.DataFrame()

In [ ]:
# -------------------------------------------------------------------
# 2. fetch_firms_all_years(sources, start_year, end_year, bbox, map_key)
# -------------------------------------------------------------------
# - Runs a nested loop over all specified satellite sources and all target archive years.
# - Calls fetch_firms_year() for each combination, applying a short delay to respect rate limits.
# - Stacks every yearly DataFrame vertically into one consolidated master DataFrame for the entire period.

def fetch_firms_all_years(sources, start_year, end_year, bbox, map_key, pause=1.0):
    frames = []
    for source in sources:
        for year in range(start_year, end_year + 1):
            print(f"Fetching {source} {year} ...")
            df = fetch_firms_year(source, year, bbox, map_key)
            if not df.empty:
                frames.append(df)
            time.sleep(pause)  # be polite to the API
    if not frames:
        raise RuntimeError("No FIRMS data retrieved — check MAP_KEY / bbox / sources.")
    return pd.concat(frames, ignore_index=True)

In [ ]:
# -------------------------------------------------------------------
# 3. standardize_firms(df)
# -------------------------------------------------------------------
# - Zero-pads satellite acquisition time strings and builds a unified, parsed acq_datetime column.
# - Filters down to required attributes, standardizes coordinate column names (lat/lon), and casts numeric FRP.
# - Drops rows with missing coordinates or datetimes to produce a clean Pandas dataset ready for GeoPandas conversion.

def standardize_firms(df):
    """Keep the attributes needed by the workflow and build a proper datetime."""
    df = df.copy()
    df["acq_time"] = df["acq_time"].astype(str).str.zfill(4)
    df["acq_datetime"] = pd.to_datetime(
        df["acq_date"] + " " + df["acq_time"].str[:2] + ":" + df["acq_time"].str[2:],
        errors="coerce",
    )
    keep_cols = ["latitude", "longitude", "acq_date", "acq_time", "acq_datetime",
                 "frp", "confidence", "satellite", "instrument", "source"]
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].rename(columns={"latitude": "lat", "longitude": "lon"})
    df["frp"] = pd.to_numeric(df["frp"], errors="coerce")
    df = df.dropna(subset=["lat", "lon", "acq_datetime", "frp"])
    return df


In [ ]:

firms_raw = fetch_firms_all_years(
    sources=FIRMS_SOURCES,
    start_year=START_YEAR,
    end_year=END_YEAR,
    bbox=basin_bbox,
    map_key=FIRMS_MAP_KEY,
)
print("Raw FIRMS records (bbox only):", firms_raw.shape)

In [ ]:
firms_std = standardize_firms(firms_raw)
firms_gdf = gpd.GeoDataFrame(
    firms_std,
    geometry=gpd.points_from_xy(firms_std["lon"], firms_std["lat"]),
    crs="EPSG:4326",
)

# Precise spatial clip to the actual basin polygon (bbox above was just coarse pre-filter)
firms_basin = gpd.sjoin(firms_gdf, basin_gdf[["basin_name", "geometry"]], predicate="within", how="inner")
firms_basin = firms_basin.drop(columns=["index_right"])

print(f"FIRMS detections in bbox: {len(firms_gdf)}  ->  within Godavari basin: {len(firms_basin)}")
firms_basin.head()

In [ ]:

# Save Stage output
firms_out_path = OUT_DIR / f"firms_godavari_{START_YEAR}_{END_YEAR}.gpkg"
firms_basin.to_file(firms_out_path, driver="GPKG")
firms_basin.drop(columns="geometry").to_csv(
    OUT_DIR / f"firms_godavarGodi_{START_YEAR}_{END_YEAR}.csv", index=False
)
print("Saved:", firms_out_path)

### Earth Engine

In [ ]:
EE_PROJECT = "fluid-blade-497705-q1"

In [ ]:
try:
    ee.Initialize(project=EE_PROJECT)
    print("EE already authenticated — initialized successfully.")
except Exception:
    print("No valid credentials found — launching browser auth...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print("EE authenticated and initialized.")

### Stage 3: ESRI 10m LULC + pilot year

In [ ]:

# ESRI 10m Annual Land Use/Land Cover (2017–2024), GEE Community Catalog
# https://gee-community-catalog.org/projects/S2TSLULC/
ESRI_LULC_COLLECTION = "projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS"

# ESRI LULC class codes (band 'b1'):
# 1 Water | 2 Trees | 4 Flooded Vegetation | 5 Crops | 7 Built Area
# 8 Bare Ground | 9 Snow/Ice | 10 Clouds | 11 Rangeland
# TODO / TO BE VALIDATED: confirm "Crops" (5) alone is the class you want,
# or whether "Rangeland" (11) should also be considered in some sub-basins.
AGRI_LULC_CLASSES = [5]

# Pilot year for Stage 3 (per proposal — test on one year before 2019-2024 full run)
PILOT_YEAR = 2021

# VIIRS-only sources (MODIS excluded per Stage 3 spec)
VIIRS_SOURCES = ["VIIRS_SNPP_SP", "VIIRS_NOAA20_SP"]

Filter to VIIRS-only, pilot year

In [ ]:
firms_basin = pd.read_csv(
    OUT_DIR / "firms_godavari_2019_2024.csv"
)

firms_basin["acq_datetime"] = pd.to_datetime(
    firms_basin["acq_datetime"]
)

# Step 1: Subset FIRMS data to VIIRS + pilot year
firms_basin["year"] = firms_basin["acq_datetime"].dt.year

viirs_pilot = firms_basin[
    (firms_basin["source"].isin(VIIRS_SOURCES)) &
    (firms_basin["year"] == PILOT_YEAR)
].reset_index(drop=True)

print(f"VIIRS detections in {PILOT_YEAR} (Godavari basin): {len(viirs_pilot)}")
viirs_pilot.head()

Load ESRI LULC image for that year

In [ ]:
basin_ee_fc = geemap.geopandas_to_ee(basin_gdf)
basin_geom = basin_ee_fc.geometry()

In [ ]:
# ============================================================
# Stage 3 — Step 2: ESRI 10m LULC image for the pilot year
# ============================================================
def get_esri_lulc_image(year, region):
    """
    Return a single mosaicked ESRI 10m LULC image (band 'b1') for the given year,
    clipped to the basin region. The source collection is tiled globally, so a
    mosaic is required to get full basin coverage.
    """
    start = f"{year}-01-01"
    end   = f"{year + 1}-01-01"

    col = (
        ee.ImageCollection(ESRI_LULC_COLLECTION)
        .filterDate(start, end)
        .filterBounds(region)
    )

    n_tiles = col.size().getInfo()
    if n_tiles == 0:
        raise ValueError(f"No ESRI LULC tiles found for {year} over this basin.")

    img = col.mosaic().select("b1").clip(region)
    print(f"ESRI LULC {year}: {n_tiles} tile(s) mosaicked.")
    return img


lulc_img_pilot = get_esri_lulc_image(PILOT_YEAR, basin_geom)

In [ ]:
Map = geemap.Map()

In [ ]:
# ============================================================
#Visualisation

# Center map on the basin
Map.centerObject(basin_geom, 7)

# Add ESRI LULC
Map.addLayer(
    lulc_img_pilot,
    {
        "min": 1,
        "max": 11,
        "palette": [
            "419BDF",  # Water
            "397D49",  # Trees
            "88B053",  # Flooded Vegetation
            "E49635",  # Crops
            "C4281B",  # Built Area
            "A59B8F",  # Bare Ground
            "B39FE1",  # Snow/Ice
            "FFFFFF",  # Clouds
            "DFC35A",  # Rangeland
        ],
    },
    f"ESRI LULC {PILOT_YEAR}"
)

# Add basin boundary
Map.addLayer(
    basin_ee_fc.style(
        color="000000",
        fillColor="00000000",
        width=2
    ),
    {},
    "Godavari Basin"
)

Map

#### Sample LULC at each fire point (batched) - Computationally Heavy 

In [ ]:
# ============================================================
# LULC sampling in 10k-point batches
# ============================================================

BATCH_SIZE = 10_000

def export_lulc_batch(points_gdf, lulc_image, description, scale=10, folder="EE_exports"):
    """
    Samples LULC server-side for one batch of FIRMS points
    and exports the result as a CSV to Google Drive.
    """
    pts = points_gdf.copy()
    # Convert only row_id + geometry to Earth Engine
    fc = geemap.geopandas_to_ee(
        pts[["row_id", "geometry"]]
    )
    # Sample categorical LULC at each fire location
    sampled_fc = lulc_image.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.first(),
        scale=scale,
    )
    # Keep only the fields needed for rejoining locally
    sampled_fc = sampled_fc.select(
        propertySelectors=["row_id", "first"],
        newProperties=["row_id", "lulc_class"],
    )
    # Export this batch
    task = ee.batch.Export.table.toDrive(
        collection=sampled_fc,
        description=description,
        folder=folder,
        fileNamePrefix=description,
        fileFormat="CSV",
    )
    task.start()
    print(f"Started: {description} | Task ID: {task.id}")
    return task


In [ ]:

# ------------------------------------------------------------
# Create stable row_id + point geometry
# ------------------------------------------------------------

viirs_pilot_indexed = viirs_pilot.reset_index(drop=True).copy()

# Stable ID used to reconnect LULC results later
viirs_pilot_indexed["row_id"] = viirs_pilot_indexed.index

# Create point geometry from FIRMS longitude/latitude
viirs_pilot_indexed_gdf = gpd.GeoDataFrame(
    viirs_pilot_indexed,
    geometry=gpd.points_from_xy(
        viirs_pilot_indexed["lon"],
        viirs_pilot_indexed["lat"]
    ),
    crs="EPSG:4326"
)

print(f"Total {PILOT_YEAR} VIIRS points: {len(viirs_pilot_indexed_gdf):,}")

n_points = len(viirs_pilot_indexed)
n_batches = (n_points + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Total {PILOT_YEAR} VIIRS points: {n_points:,}")
print(f"Batch size: {BATCH_SIZE:,}")
print(f"Number of batches: {n_batches}")

In [ ]:
# ------------------------------------------------------------
# Submit batches
# ------------------------------------------------------------

BATCH_SIZE = 10_000
tasks = []
for batch_no, start in enumerate(
    range(0, len(viirs_pilot_indexed_gdf), BATCH_SIZE)
):
    end = min(
        start + BATCH_SIZE,
        len(viirs_pilot_indexed_gdf)
    )
    batch = viirs_pilot_indexed_gdf.iloc[start:end].copy()
    description = (
        f"lulc_sample_godavari_{PILOT_YEAR}_c{batch_no:03d}"
    )
    task = export_lulc_batch(
        batch,
        lulc_img_pilot,
        description=description,
        scale=10,
        folder="EE_exports",
    )
    tasks.append(task)
    print(
        f"Batch {batch_no:03d}: "
        f"rows {start:,}–{end-1:,} "
        f"({len(batch):,} points)"
    )

print(f"\nSubmitted {len(tasks)} GEE export tasks.")

save those GEE exports locally in data/River_Name/LULC_fire/

In [ ]:
# Merge downloaded LULC batch CSVs

LULC_FIRE_DIR = DATA_DIR / "LULC_fire"

batch_files = sorted(
    LULC_FIRE_DIR.glob(
        f"lulc_sample_godavari_{PILOT_YEAR}_c*.csv"
    )
)
print(f"Found {len(batch_files)} LULC batch CSVs.")

if len(batch_files) == 0:
    raise FileNotFoundError(
        f"No LULC batch CSVs found in: {LULC_FIRE_DIR}"
    )

In [ ]:
# Merge all batch CSVs
lulc_df = pd.concat(
    [pd.read_csv(f) for f in batch_files],
    ignore_index=True
)[["row_id", "lulc_class"]]

print(f"Total LULC records merged: {len(lulc_df):,}")
print(f"Original VIIRS points: {len(viirs_pilot_indexed_gdf):,}")
print(f"LULC results returned: {len(lulc_df):,}")
print(f"Unique row_ids returned: {lulc_df['row_id'].nunique():,}")

In [ ]:
# Join LULC results back to the original VIIRS detections
viirs_pilot_lulc = (
    viirs_pilot_indexed_gdf
    .set_index("row_id")
    .join(
        lulc_df.set_index("row_id"),
        how="left"
    )
    .reset_index(drop=True)
)

print(f"Final joined dataset: {len(viirs_pilot_lulc):,} rows")

viirs_pilot_lulc.head()

In [ ]:
# Save the LULC-joined VIIRS dataset
lulc_joined_out = OUT_DIR / f"viirs_lulc_godavari_{PILOT_YEAR}.csv"

viirs_pilot_lulc.to_csv(lulc_joined_out, index=False)

print(f"Saved: {lulc_joined_out}")

In [ ]:
# Load saved LULC-joined VIIRS data
lulc_joined_out = OUT_DIR / f"viirs_lulc_godavari_{PILOT_YEAR}.csv"

viirs_pilot_lulc = pd.read_csv(lulc_joined_out)

print(f"Loaded: {lulc_joined_out}")
print(f"VIIRS detections: {len(viirs_pilot_lulc):,}")

# Keep valid LULC classes
viirs_pilot_lulc = viirs_pilot_lulc.dropna(subset=["lulc_class"])
viirs_pilot_lulc["lulc_class"] = viirs_pilot_lulc["lulc_class"].astype(int)

# Retain agricultural fires only
firms_agri = viirs_pilot_lulc[
    viirs_pilot_lulc["lulc_class"].isin(AGRI_LULC_CLASSES)
].copy()

print(
    f"Agricultural fires: {len(firms_agri):,} "
    f"({100 * len(firms_agri) / len(viirs_pilot_lulc):.1f}%)"
)

final_cols = [
    "acq_date", "acq_time", "acq_datetime",
    "lat", "lon", "frp", "satellite",
    "confidence", "instrument", "source", "lulc_class"
]
final_cols = [c for c in final_cols if c in firms_agri.columns]

firms_agri = firms_agri[final_cols]

agri_out_path = OUT_DIR / f"viirs_agri_godavari_{PILOT_YEAR}.csv"
firms_agri.to_csv(agri_out_path, index=False)

print("Saved:", agri_out_path)

firms_agri.head()

### Only run this cell, instead of re-running previous cells for a year.

In [ ]:

# plausibility check

# Load saved agricultural-fire dataset
agri_out_path = OUT_DIR / f"viirs_agri_godavari_{PILOT_YEAR}.csv"
firms_agri = pd.read_csv(agri_out_path)
print(f"Loaded: {agri_out_path}")
print(f"Agricultural VIIRS fires: {len(firms_agri):,}")

# Create point geometry for spatial check
firms_agri_gdf = gpd.GeoDataFrame(
    firms_agri,
    geometry=gpd.points_from_xy(
        firms_agri["lon"],
        firms_agri["lat"]
    ),
    crs="EPSG:4326",
)
# Plot
fig, ax = plt.subplots(figsize=(7, 7))
basin_gdf.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=1
)
firms_agri_gdf.plot(
    ax=ax,
    column="frp",
    cmap="hot_r",
    markersize=4,
    legend=True,
    alpha=0.6
)
ax.set_title(
    f"Agricultural VIIRS Fires: Godavari Basin ({PILOT_YEAR})"
)
plt.tight_layout()
plt.show()

In [ ]:
# Quick analysis: monthly fire activity and FRP
firms_agri["month"] = pd.to_datetime(firms_agri["acq_datetime"]).dt.month

monthly = firms_agri.groupby("month").agg(
    fire_detections=("frp", "size"),
    mean_frp=("frp", "mean"),
    max_frp=("frp", "max")
).round(2)

print(monthly)